In [1]:
import torch
import numpy as np
import h5py
from text_embeddings.embeddings import HFBertExtractor

/home/felipefg/dsc/CPE883-2025-02/experiments/text_embeddings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = '/home/felipefg/dsc/experiments/data/bert_smaller_nano_twods_v1/checkpoint-18000'

In [3]:
emb = HFBertExtractor(model_path)

In [4]:
emb.load_model()

In [5]:
texts = [
    ' The second entry , a " commercial , yet heavy " album called Addicted , was released in November 2009 and features lead vocals from Townsend and Dutch singer Anneke van Giersbergen . Brian " Beav " Waddell was recruited from the Devin Townsend Band to play bass .',
    'The suspect of [MASK] shooting was captured. The president said he wants a fast trial.',
]

In [6]:
embs = emb.get_embeddings_batch(texts)

/home/felipefg/dsc/CPE883-2025-02/experiments/text_embeddings/.venv/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [7]:
embs.shape

torch.Size([2, 192])

In [8]:
import hf_dataset_loader

In [9]:
loader = hf_dataset_loader.TwitterFinancialNewsTopicDataset()

In [10]:
for record in loader.dataset['train']:
    print(record)
    break

{'text': "Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more  ", 'label': 0}


In [32]:
dataset = loader.dataset['train']

In [34]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 16990
})

In [12]:
n_samples = len(dataset)
n_samples

16990

In [13]:
batch_size = 30

In [14]:
embedding_size = emb.model.config.hidden_size

In [15]:
embeddings = np.zeros((n_samples, embedding_size))

In [35]:
texts = np.empty(n_samples, dtype=object)
labels = np.empty(n_samples, dtype=int)

In [36]:
batch_size = 32
for start_idx in range(0, n_samples, batch_size):
    end_idx = min(start_idx + batch_size, n_samples)
    batch_docs = dataset[start_idx:end_idx]

    batch_embeddings = emb.get_embeddings_batch(batch_docs['text'])
    
    # Fill pre-allocated arrays
    embeddings[start_idx:end_idx, :] = batch_embeddings.numpy()
    texts[start_idx:end_idx] = batch_docs['text']
    labels[start_idx:end_idx] = batch_docs['label']

/home/felipefg/dsc/CPE883-2025-02/experiments/text_embeddings/.venv/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [19]:
batch_embeddings.shape

torch.Size([2, 192])

In [20]:
start_idx, end_idx

(0, 32)

In [31]:
dataset['text']

Column(["Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more  ", 'Buy Las Vegas Sands as travel to Singapore builds, Wells Fargo says  ', 'Piper Sandler downgrades DocuSign to sell, citing elevated risks amid CEO transition  ', "Analysts react to Tesla's latest earnings, break down what's next for electric car maker  ", 'Netflix and its peers are set for a ‘return to growth,’ analysts say, giving one stock 120% upside  '])

In [38]:
with h5py.File('results.h5', 'w') as fout:
    fout.create_dataset('embeddings', data=embeddings)
    fout.create_dataset('texts', data=texts)
    fout.create_dataset('labels', data=labels)

In [39]:
!ls -lh

total 21M
-rwxrwxr-x 1 felipefg felipefg  528 Sep 12 15:10 build_image.sh
drwxrwxr-x 3 felipefg felipefg    3 Sep 12 16:41 data
-rw-rw-r-- 1 felipefg felipefg  474 Sep 12 15:10 Dockerfile
-rw-rw-r-- 1 felipefg felipefg  657 Sep 15 08:52 pyproject.toml
-rw-rw-r-- 1 felipefg felipefg 8.0K Sep 12 15:10 README.md
-rw-rw-r-- 1 felipefg felipefg  28M Sep 15 09:20 results.h5
drwxrwxr-x 3 felipefg felipefg    7 Sep 15 09:02 text_embeddings
-rw-rw-r-- 1 felipefg felipefg 8.6K Sep 15 09:19 Untitled.ipynb
-rw-rw-r-- 1 felipefg felipefg 164K Sep 15 08:52 uv.lock


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [41]:
dataset.__class__

datasets.arrow_dataset.Dataset